In [ ]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, average_precision_score
from pyod.utils.data import precision_n_scores
from pyod.models.iforest import IForest
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from pyod.models.xgbod import XGBOD
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

In [ ]:
def evaluate_metrics(y_test, y_pred, y_proba=None, digits=3):
    res = {"Accuracy": round(accuracy_score(y_test, y_pred), digits),
           "Precision": round(precision_score(y_test, y_pred), digits),
           "Recall": round(recall_score(y_test, y_pred), digits),
           "F1": round(f1_score(y_test, y_pred), digits),
           "MCC": round(matthews_corrcoef(y_test, y_pred), ndigits=digits)}
    if y_proba is not None:
        res["AUC_PR"] = round(average_precision_score(y_test, y_proba), digits)
        res["AUC_ROC"] = round(roc_auc_score(y_test, y_proba), digits)
        res["PREC_N_SCORES"] = round(precision_n_scores(y_test, y_proba), digits)
    return res


def set_seed_numpy(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
features = [
    "mean", "var", "std", "len", "duration", "len_weighted", "gaps_squared", "n_peaks",
    "smooth10_n_peaks", "smooth20_n_peaks", "var_div_duration", "var_div_len",
    "diff_peaks", "diff2_peaks", "diff_var", "diff2_var", "kurtosis", "skew",
]
SEED = 2137


In [ ]:
df = pd.read_csv("data/dataset.csv", index_col="segment")

X_train, y_train = df.loc[df.train==1, features], df.loc[df.train==1, "anomaly"]

X_test, y_test = df.loc[df.train==0, features], df.loc[df.train==0, "anomaly"]

X_train_nominal = df.loc[(df.anomaly==0)&(df.train==1), features]

#standardize the data to have a mean of 0 and a standard deviation of 1
#useful when values are in different units, varying length, and sampling frequency
prep = StandardScaler()
X_train_nominal2 = prep.fit_transform(X_train_nominal)
X_train2 = prep.transform(X_train)
X_test2 = prep.transform(X_test)

In [ ]:
set_seed_numpy(SEED)

In [ ]:
# supervised example

In [ ]:
import shap

In [ ]:
shap.initjs()

In [ ]:
model = AdaBoostClassifier(random_state=SEED)
model.fit(X_train2, y_train)

y_predicted = model.predict(X_test2)
y_predicted_score = model.decision_function(X_test2)

print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
print("y_predicted (first 60):", y_predicted[:60])
print("y_predicted_score (first 60):", y_predicted_score[:60])

In [ ]:
print(df['anomaly'].value_counts())

In [ ]:
X_explain = X_test2[:100]

explainer = shap.KernelExplainer(
    model.decision_function, shap.sample(X_train2, 100, random_state=SEED)
)

shap_values = explainer(X_explain)

In [ ]:
shap.summary_plot(shap_values, X_explain)
shap.waterfall_plot(shap_values[27])
shap.plots.force(shap_values[27])
#shap.plots.force(shap_values)

In [ ]:
i = 27
print("SHAP values for sample 0:", shap_values.values[i])

print("\nBase value:", shap_values.base_values[i])
print("Sum of SHAP values:", shap_values.values[i].sum())
print("Base + sum(SHAP):", shap_values.base_values[i] + shap_values.values[i].sum())

In [ ]:
y_predicted_explain = model.predict(X_explain)

anomaly_mask = (y_predicted_explain == 1)
normal_mask  = (y_predicted_explain == 0)

shap_anom = shap_values.values[anomaly_mask]
shap_norm = shap_values.values[normal_mask]

# how strongly a feature influences a model's prediction
anom_importance = np.mean(np.abs(shap_anom), axis=0)
norm_importance = np.mean(np.abs(shap_norm), axis=0)

# what direction is the feature pushing the prediction toward(positive or negative)
anom_direction = np.mean(shap_anom, axis=0)
norm_direction = np.mean(shap_norm, axis=0)


df_attack_features = pd.DataFrame({
    "feature": features,
    "anomaly_mean_shap": anom_direction,
    "normal_mean_shap": norm_direction,
    "difference": anom_direction - norm_direction,
    "anomaly_abs_importance": anom_importance
}).sort_values("difference") 

print("Features Driving Anomalous Behavior (most → least):")
print(df_attack_features.to_string(index=False))

In [ ]:
# unsupervised example

In [ ]:
#model = IForest(random_state=SEED)
#model.fit(X_train2)

#y_predicted = model.predict(X_test2)
#y_predicted_score = model.decision_function(X_test2)

#print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
model = IForest(random_state=SEED, contamination=.2)
model.fit(X_train2)

y_predicted = model.predict(X_test2)
y_predicted_score = model.decision_function(X_test2)

print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))


In [ ]:
print("y_predicted (first 10):", y_predicted[:60])
print("y_predicted_score (first 10):", y_predicted_score[:60])

In [ ]:
print(df['anomaly'].value_counts())

In [ ]:
explainer = shap.Explainer(model)
shap_values = explainer(X_test2)

shap.summary_plot(shap_values, X_test2)
shap.waterfall_plot(shap_values[58])

shap.plots.force(shap_values[58])

In [ ]:
i = 27
print("SHAP values for sample 0:", shap_values.values[i])

print("\nBase value:", shap_values.base_values[i])
print("Sum of SHAP values:", shap_values.values[i].sum())
print("Base + sum(SHAP):", shap_values.base_values[i] + shap_values.values[i].sum())

In [ ]:
anomaly_mask = y_predicted == 1
normal_mask  = y_predicted == 0

shap_anom = shap_values.values[anomaly_mask]
shap_norm = shap_values.values[normal_mask]

# how strongly a feature influences a model's prediction
anom_importance = np.mean(np.abs(shap_anom), axis=0)
norm_importance = np.mean(np.abs(shap_norm), axis=0)


# what direction is the feature pushing the prediction toward(positive or negative)
anom_direction = np.mean(shap_anom, axis=0)
norm_direction = np.mean(shap_norm, axis=0)


df_attack_features = pd.DataFrame({
    "feature": features,
    "anomaly_mean_shap": anom_direction,
    "normal_mean_shap": norm_direction,
    "difference": anom_direction - norm_direction,
    "anomaly_abs_importance": anom_importance
}).sort_values("difference")

print("Features Driving Anomalous Behavior (most → least):")
print(df_attack_features.to_string(index=False))


In [ ]:
#model = RandomForestClassifier(random_state=SEED)
#model.fit(X_train2,y_train)

#y_predicted = model.predict(X_test2)
#y_predicted_score = model.predict_proba(X_test2)[:, 1]

#print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))
#y_predicted


In [ ]:
#explainer = shap.TreeExplainer(model)
#shap_values = explainer.shap_values(X_test2, check_additivity=False)


#shap.summary_plot(shap_values, X_test2)
#shap.plots.bar(shap_values)

In [ ]:
#model = XGBOD(random_state=SEED, contamination=.2)
#model.fit(X_train2, y_train)

#y_predicted = model.predict(X_test2)
#y_predicted_score = model.decision_function(X_test2)

#print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
#X_explain = X_test2[:50]

#explainer = shap.Explainer(
#    model.decision_function,
#    shap.sample(X_train2, 100, random_state=SEED)
#)

#shap_values = explainer(X_explain)


In [ ]:
#np.shape(shap_values)
#shap.summary_plot(shap_values, X_explain)
#shap.waterfall_plot(shap_values[0])
#shap.plots.bar(shap_values)

#shap.plots.force(shap_values)

In [ ]:
model = LogisticRegression(random_state=SEED)
model.fit(X_train2, y_train)

y_predicted = model.predict(X_test2)
y_predicted_score = model.decision_function(X_test2)

print(model, '\n', evaluate_metrics(y_test, y_predicted, y_predicted_score))

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train2.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from sklearn.utils import class_weight
import numpy as np

class_weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

In [ ]:
model.fit(
    X_train2,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights
)

In [ ]:
y_predicted = model.predict(X_test2)
y_predicted_score = (y_predicted > 0.5).astype(int)

print(model, '\n', evaluate_metrics(y_test, y_predicted_score, y_predicted))